# CURE-Rec — reviewer-revision experiments

This notebook contains **new evidence**, not a rerun of accepted results. Every expensive action is disabled by default. It addresses held-out portfolio evaluation and selector baselines without inventing values.


## Setup


In [ ]:
from pathlib import Path
import importlib
import sys

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the CURE-Rec code directory or repository root.')
sys.path[:] = [str(ROOT), *[item for item in sys.path if item != str(ROOT)]]
for name in list(sys.modules):
    if name == 'cure_rec' or name.startswith('cure_rec.'):
        del sys.modules[name]
importlib.invalidate_caches()

from cure_rec.config import load_settings
from cure_rec.revision import recompute_selector_summary, run_selector_holdout_study

FULL_CONFIG = ROOT / 'configs' / 'curesim_full.yaml'
RUN_ROOT = ROOT / 'runs'
print('CURE-Rec source:', ROOT)


## Action 1 — held-out portfolio-selection benchmark

This is the reviewer-required experiment. Portfolios are selected on `SELECTION_SEEDS`, frozen, then evaluated on disjoint `EVALUATION_SEEDS`. It compares exact CURE maximin against singleton, robust-Shapley, greedy, nominal-only, random-feasible, and grand-coalition diagnostic selectors.

**Cost:** every selection/evaluation seed executes an exact 64-coalition game. Start with the smoke configuration, inspect the files, then deliberately enable the full configuration.


In [ ]:
RUN_SELECTOR_HOLDOUT_SMOKE = False
RUN_SELECTOR_HOLDOUT_FULL = False

assert not (RUN_SELECTOR_HOLDOUT_SMOKE and RUN_SELECTOR_HOLDOUT_FULL)

if RUN_SELECTOR_HOLDOUT_SMOKE:
    cfg = load_settings(ROOT / 'configs' / 'curesim_quickstart.yaml')
    cfg.run.output_root = RUN_ROOT
    revision_run = run_selector_holdout_study(
        cfg, selection_seeds=(42,), evaluation_seeds=(200, 201)
    )
    print('Selector holdout smoke run:', revision_run)
elif RUN_SELECTOR_HOLDOUT_FULL:
    cfg = load_settings(FULL_CONFIG)
    cfg.run.output_root = RUN_ROOT
    revision_run = run_selector_holdout_study(
        cfg, selection_seeds=(42, 43, 44, 45, 46), evaluation_seeds=tuple(range(200, 220))
    )
    print('Selector holdout full run:', revision_run)
else:
    print('Selector holdout study disabled. Enable smoke first; full run is expensive.')


## Action 2 — inspect completed revision output

Set the exact run directory printed by Action 1. This is cheap. The summary contains held-out robust utility, feasibility, paired differences versus exact CURE maximin, standardized paired effect, and an exact sign-test p-value.


In [ ]:
import pandas as pd
REVISION_OUTPUT = None

if REVISION_OUTPUT is not None:
    REVISION_OUTPUT = Path(REVISION_OUTPUT)
    display(pd.read_csv(REVISION_OUTPUT / 'selection_choices.csv'))
    display(recompute_selector_summary(REVISION_OUTPUT))
    display(pd.read_csv(REVISION_OUTPUT / 'heldout_selector_evaluations.csv').head())
else:
    print('Set REVISION_OUTPUT to inspect a completed selector study.')


## Required interpretation

The held-out selector study is simulator-conditional evidence. It does not turn MovieLens into causal data. Do not replace the current manuscript values with these results until the full run, archive, and manuscript tables have been reviewed.
